# Landslide Risk Specialist + Leader LLM — Training Notebook

Two-tier proof of concept for Envis's landslide hazard system:

1. **Specialist**: a small tabular transformer (FT-Transformer style, Gorishniy et al. 2021) trained
   to predict landslide risk from rainfall/slope/soil features. Trains in a few minutes on a free T4.
2. **Leader**: an LLM that calls the specialist as a tool and produces a fused, human-readable risk
   explanation (tool-calling demo).

**Ground truth for training data** is generated from Envis's own production formulas — not arbitrary
synthetic labels:
- Caine (1980) global rainfall intensity–duration threshold: `I = 14.82 * D^-0.39`
- Montgomery & Dietrich (1994) infinite-slope factor of safety (the physics behind USGS TRIGRS/SHALSTAB),
  exactly as implemented in `src/lib/meteorology.ts` (`infiniteSlopeFactorOfSafety`, `soilMechanicalParams`)

**Runtime:** In Colab, set `Runtime > Change runtime type > T4 GPU` before running.


## 0. Setup

In [ ]:
import torch, random, json, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE, "-", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU found")

import os
os.makedirs("plots", exist_ok=True)
os.makedirs("artifacts", exist_ok=True)


## 1. Physics-grounded synthetic dataset

Ported 1:1 from Envis's `src/lib/meteorology.ts` so the specialist is learning a real, published
hazard model rather than an arbitrary function.


In [ ]:
SOIL_TEXTURES = {
    "Clay":            {"cohesionKPa": 10,  "frictionAngleDeg": 20, "unitWeightKNm3": 19.0},
    "Clay loam":       {"cohesionKPa": 8,   "frictionAngleDeg": 24, "unitWeightKNm3": 18.5},
    "Sandy clay loam": {"cohesionKPa": 5,   "frictionAngleDeg": 27, "unitWeightKNm3": 18.0},
    "Loam":            {"cohesionKPa": 4,   "frictionAngleDeg": 28, "unitWeightKNm3": 18.0},
    "Silt loam":       {"cohesionKPa": 3,   "frictionAngleDeg": 27, "unitWeightKNm3": 17.5},
    "Silt":            {"cohesionKPa": 2,   "frictionAngleDeg": 26, "unitWeightKNm3": 17.0},
    "Loamy sand":      {"cohesionKPa": 1,   "frictionAngleDeg": 32, "unitWeightKNm3": 17.5},
    "Sand":            {"cohesionKPa": 0.5, "frictionAngleDeg": 33, "unitWeightKNm3": 17.0},
}
TEXTURE_NAMES = list(SOIL_TEXTURES.keys())
GAMMA_WATER = 9.81  # kN/m^3
CAINE_DURATIONS = [1, 3, 6, 12, 24]  # hours


def caine_threshold_mmh(duration_h: float) -> float:
    """Caine (1980) global rainfall intensity-duration threshold, mm/h."""
    return 14.82 * (duration_h ** -0.39)


def infinite_slope_fs(slope_deg, cohesion_kpa, friction_deg, unit_weight_knm3, soil_depth_m, saturation_fraction):
    """Montgomery & Dietrich (1994) infinite-slope factor of safety.
    FS = [c\' + (gamma*z - gamma_w*m*z)*cos^2(beta)*tan(phi\')] / (gamma*z*sin(beta)*cos(beta))
    """
    beta = math.radians(max(0.5, slope_deg))
    phi = math.radians(friction_deg)
    z = max(0.1, soil_depth_m)
    m = min(1.0, max(0.0, saturation_fraction))
    gamma = unit_weight_knm3

    driving = gamma * z * math.sin(beta) * math.cos(beta)
    if driving <= 0:
        return 10.0
    resisting = cohesion_kpa + (gamma * z - GAMMA_WATER * m * z) * math.cos(beta) ** 2 * math.tan(phi)
    return max(0.0, resisting / driving)


def clamp01(x):
    return min(1.0, max(0.0, x))


def risk_class_from_score(score: float) -> str:
    if score >= 0.75:
        return "extreme"
    if score >= 0.5:
        return "high"
    if score >= 0.25:
        return "moderate"
    return "low"


def generate_sample(rng: random.Random):
    """One synthetic (features, ground-truth label) pair, matching the
    /api/forecast-risk landslide scoring pipeline."""
    # --- terrain / soil ---
    slope_deg = rng.uniform(0.5, 50.0)
    texture = rng.choice(TEXTURE_NAMES)
    params = SOIL_TEXTURES[texture]
    soil_depth_m = max(0.4, rng.gauss(1.2, 0.25))  # Envis assumes 1.2m shallow regolith

    # --- rainfall: a random "storm regime" decaying with duration, plus noise,
    #     so intensities are physically plausible but not perfectly collinear ---
    storm_intensity = rng.lognormvariate(mu=2.0, sigma=0.9)  # heavy-tailed, mm/h at D=1h scale
    decay_exp = rng.uniform(0.25, 0.55)
    rain = {}
    for D in CAINE_DURATIONS:
        base = storm_intensity * (D ** -decay_exp)
        noisy = max(0.0, base * rng.lognormvariate(0, 0.15))
        rain[f"rain_{D}h_mmh"] = noisy

    # --- soil saturation: correlated with the longer-duration rainfall + a random
    #     antecedent-moisture baseline, not independent noise ---
    api_baseline = rng.betavariate(2, 3)  # antecedent precipitation index proxy, skewed low
    rain_driven = clamp01(rain["rain_24h_mmh"] / 25.0)
    saturation_fraction = clamp01(0.6 * rain_driven + 0.4 * api_baseline + rng.gauss(0, 0.05))

    # --- Envis scoring pipeline ---
    ratios = [rain[f"rain_{D}h_mmh"] / caine_threshold_mmh(D) for D in CAINE_DURATIONS]
    trigger_f = clamp01(max(ratios))

    fs = infinite_slope_fs(slope_deg, params["cohesionKPa"], params["frictionAngleDeg"],
                            params["unitWeightKNm3"], soil_depth_m, saturation_fraction)
    stability_f = clamp01((1.6 - fs) / 1.1)

    saturation_f = saturation_fraction  # Envis blends soil-moisture + API; we already did that above

    score = trigger_f * 0.4 + stability_f * 0.35 + saturation_f * 0.25
    # small measurement/label noise so the task isn't a trivial deterministic fit
    score_noisy = clamp01(score + rng.gauss(0, 0.03))

    features = {
        "slope_deg": slope_deg,
        "soil_texture": texture,
        "soil_depth_m": soil_depth_m,
        "saturation_fraction": saturation_fraction,
        **rain,
    }
    label = {
        "factor_of_safety": fs,
        "trigger_f": trigger_f,
        "stability_f": stability_f,
        "risk_score": score_noisy,
        "risk_class": risk_class_from_score(score_noisy),
    }
    return features, label


def build_dataset(n: int, seed: int = SEED) -> pd.DataFrame:
    rng = random.Random(seed)
    rows = []
    for _ in range(n):
        feats, label = generate_sample(rng)
        rows.append({**feats, **label})
    return pd.DataFrame(rows)


N_SAMPLES = 40000
df = build_dataset(N_SAMPLES)
df.to_csv("artifacts/landslide_synthetic_dataset.csv", index=False)
print(df.shape)
df.head()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(df["risk_score"], bins=40, color="#4c72b0")
axes[0].set_title("Risk score distribution")
axes[0].set_xlabel("risk_score"); axes[0].set_ylabel("count")

df["risk_class"].value_counts().reindex(["low", "moderate", "high", "extreme"]).plot(
    kind="bar", ax=axes[1], color="#dd8452")
axes[1].set_title("Risk class balance")
plt.tight_layout()
plt.savefig("plots/00_dataset_overview.png", dpi=150)
plt.show()


## 2. Specialist model — FT-Transformer (tabular)

Each feature becomes a token via a learned per-feature linear projection (+ a `[CLS]` token), then a
small Transformer encoder attends across features. This is the standard strong baseline for tabular deep
learning (Gorishniy et al., *"Revisiting Deep Learning Models for Tabular Data"*, NeurIPS 2021) — a much
better fit than a causal-LM decoder for a pure numeric predictor.

Two heads off the `[CLS]` output: a regression head (risk_score, 0–1) and a 4-way classification head
(risk_class), trained jointly.


In [ ]:
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn

NUMERIC_COLS = ["slope_deg", "soil_depth_m", "saturation_fraction",
                 "rain_1h_mmh", "rain_3h_mmh", "rain_6h_mmh", "rain_12h_mmh", "rain_24h_mmh"]
CATEGORICAL_COLS = ["soil_texture"]
CLASS_NAMES = ["low", "moderate", "high", "extreme"]

train_df = df.sample(frac=0.7, random_state=SEED)
rest_df = df.drop(train_df.index)
val_df = rest_df.sample(frac=0.5, random_state=SEED)
test_df = rest_df.drop(val_df.index)
print(f"train={len(train_df)} val={len(val_df)} test={len(test_df)}")

num_mean = train_df[NUMERIC_COLS].mean()
num_std = train_df[NUMERIC_COLS].std().clip(lower=1e-6)
texture_to_idx = {t: i for i, t in enumerate(TEXTURE_NAMES)}
class_to_idx = {c: i for i, c in enumerate(CLASS_NAMES)}


class LandslideDataset(Dataset):
    def __init__(self, frame: pd.DataFrame):
        num = ((frame[NUMERIC_COLS] - num_mean) / num_std).values.astype("float32")
        self.numeric = torch.tensor(num)
        self.texture_idx = torch.tensor(frame["soil_texture"].map(texture_to_idx).values, dtype=torch.long)
        self.score = torch.tensor(frame["risk_score"].values.astype("float32"))
        self.cls_idx = torch.tensor(frame["risk_class"].map(class_to_idx).values, dtype=torch.long)

    def __len__(self):
        return len(self.score)

    def __getitem__(self, i):
        return self.numeric[i], self.texture_idx[i], self.score[i], self.cls_idx[i]


BATCH_SIZE = 256
train_loader = DataLoader(LandslideDataset(train_df), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(LandslideDataset(val_df), batch_size=BATCH_SIZE)
test_loader = DataLoader(LandslideDataset(test_df), batch_size=BATCH_SIZE)


class FTTransformerLite(nn.Module):
    def __init__(self, n_numeric, n_texture_classes, d_model=32, n_heads=4, n_layers=2,
                 d_ff=64, n_risk_classes=4, dropout=0.1):
        super().__init__()
        self.numeric_tokenizers = nn.ModuleList([nn.Linear(1, d_model) for _ in range(n_numeric)])
        self.texture_embedding = nn.Embedding(n_texture_classes, d_model)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        nn.init.normal_(self.cls_token, std=0.02)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
            dropout=dropout, batch_first=True, activation="gelu")
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

        self.regression_head = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, 1))
        self.classification_head = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, n_risk_classes))

    def forward(self, numeric, texture_idx):
        tokens = [tok(numeric[:, i:i + 1]) for i, tok in enumerate(self.numeric_tokenizers)]
        tokens.append(self.texture_embedding(texture_idx))
        tokens = torch.stack(tokens, dim=1)  # (B, n_features, d_model)

        cls = self.cls_token.expand(tokens.size(0), -1, -1)
        seq = torch.cat([cls, tokens], dim=1)

        encoded = self.encoder(seq)
        cls_out = encoded[:, 0]

        risk_score = torch.sigmoid(self.regression_head(cls_out)).squeeze(-1)
        risk_logits = self.classification_head(cls_out)
        return risk_score, risk_logits


model = FTTransformerLite(n_numeric=len(NUMERIC_COLS), n_texture_classes=len(TEXTURE_NAMES)).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"Specialist parameter count: {n_params:,}")


In [ ]:
EPOCHS = 40
LR = 2e-3

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
mse_loss = nn.MSELoss()
ce_loss = nn.CrossEntropyLoss()

history = {"train_loss": [], "val_loss": []}
best_val = float("inf")

for epoch in range(EPOCHS):
    model.train()
    train_losses = []
    for numeric, texture_idx, score, cls_idx in train_loader:
        numeric, texture_idx = numeric.to(DEVICE), texture_idx.to(DEVICE)
        score, cls_idx = score.to(DEVICE), cls_idx.to(DEVICE)

        pred_score, pred_logits = model(numeric, texture_idx)
        loss = mse_loss(pred_score, score) + 0.5 * ce_loss(pred_logits, cls_idx)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())
    scheduler.step()

    model.eval()
    val_losses = []
    with torch.no_grad():
        for numeric, texture_idx, score, cls_idx in val_loader:
            numeric, texture_idx = numeric.to(DEVICE), texture_idx.to(DEVICE)
            score, cls_idx = score.to(DEVICE), cls_idx.to(DEVICE)
            pred_score, pred_logits = model(numeric, texture_idx)
            loss = mse_loss(pred_score, score) + 0.5 * ce_loss(pred_logits, cls_idx)
            val_losses.append(loss.item())

    tl, vl = np.mean(train_losses), np.mean(val_losses)
    history["train_loss"].append(tl)
    history["val_loss"].append(vl)
    if vl < best_val:
        best_val = vl
        torch.save(model.state_dict(), "artifacts/specialist_best.pt")
    if epoch % 5 == 0 or epoch == EPOCHS - 1:
        print(f"epoch {epoch:3d}  train_loss {tl:.4f}  val_loss {vl:.4f}")

model.load_state_dict(torch.load("artifacts/specialist_best.pt"))
print("Loaded best checkpoint, val_loss =", best_val)


In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(history["train_loss"], label="train")
plt.plot(history["val_loss"], label="val")
plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend(); plt.title("Training curve")
plt.tight_layout()
plt.savefig("plots/01_training_curve.png", dpi=150)
plt.show()


## 3. Evaluation on held-out test set

In [ ]:
from sklearn.metrics import (mean_absolute_error, r2_score, roc_auc_score, roc_curve,
                              confusion_matrix, accuracy_score, f1_score)

model.eval()
all_true_score, all_pred_score, all_true_cls, all_pred_cls = [], [], [], []
with torch.no_grad():
    for numeric, texture_idx, score, cls_idx in test_loader:
        numeric, texture_idx = numeric.to(DEVICE), texture_idx.to(DEVICE)
        pred_score, pred_logits = model(numeric, texture_idx)
        all_true_score.append(score.numpy())
        all_pred_score.append(pred_score.cpu().numpy())
        all_true_cls.append(cls_idx.numpy())
        all_pred_cls.append(pred_logits.argmax(-1).cpu().numpy())

y_true_score = np.concatenate(all_true_score)
y_pred_score = np.concatenate(all_pred_score)
y_true_cls = np.concatenate(all_true_cls)
y_pred_cls = np.concatenate(all_pred_cls)

mae = mean_absolute_error(y_true_score, y_pred_score)
r2 = r2_score(y_true_score, y_pred_score)

y_true_binary = (y_true_score >= 0.5).astype(int)   # "high risk" (high or extreme)
y_pred_prob_binary = y_pred_score
auc = roc_auc_score(y_true_binary, y_pred_prob_binary)

acc = accuracy_score(y_true_cls, y_pred_cls)
f1 = f1_score(y_true_cls, y_pred_cls, average="macro")

specialist_metrics = {"mae": float(mae), "r2": float(r2), "roc_auc": float(auc),
                       "class_accuracy": float(acc), "class_macro_f1": float(f1)}
print(json.dumps(specialist_metrics, indent=2))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# 1. predicted vs actual
axes[0].scatter(y_true_score, y_pred_score, s=4, alpha=0.25, color="#4c72b0")
axes[0].plot([0, 1], [0, 1], "k--", lw=1)
axes[0].set_xlabel("true risk_score"); axes[0].set_ylabel("predicted risk_score")
axes[0].set_title(f"Predicted vs. actual (R\u00b2={r2:.3f}, MAE={mae:.3f})")

# 2. ROC curve
fpr, tpr, _ = roc_curve(y_true_binary, y_pred_prob_binary)
axes[1].plot(fpr, tpr, color="#dd8452", label=f"AUC={auc:.3f}")
axes[1].plot([0, 1], [0, 1], "k--", lw=1)
axes[1].set_xlabel("false positive rate"); axes[1].set_ylabel("true positive rate")
axes[1].set_title("ROC — high risk (\u2265 0.5) vs. not"); axes[1].legend()

# 3. confusion matrix
cm = confusion_matrix(y_true_cls, y_pred_cls, labels=list(range(4)))
im = axes[2].imshow(cm, cmap="Blues")
axes[2].set_xticks(range(4)); axes[2].set_xticklabels(CLASS_NAMES, rotation=45)
axes[2].set_yticks(range(4)); axes[2].set_yticklabels(CLASS_NAMES)
axes[2].set_xlabel("predicted"); axes[2].set_ylabel("true")
axes[2].set_title(f"Confusion matrix (acc={acc:.3f})")
for i in range(4):
    for j in range(4):
        axes[2].text(j, i, cm[i, j], ha="center", va="center",
                      color="white" if cm[i, j] > cm.max() / 2 else "black")

plt.tight_layout()
plt.savefig("plots/02_specialist_results.png", dpi=150)
plt.show()


In [ ]:
# calibration curve: bucket predicted probability, compare to observed frequency
bins = np.linspace(0, 1, 11)
bin_idx = np.digitize(y_pred_prob_binary, bins) - 1
bin_idx = np.clip(bin_idx, 0, 9)
observed, predicted_mean = [], []
for b in range(10):
    mask = bin_idx == b
    if mask.sum() == 0:
        continue
    observed.append(y_true_binary[mask].mean())
    predicted_mean.append(y_pred_prob_binary[mask].mean())

plt.figure(figsize=(5, 5))
plt.plot([0, 1], [0, 1], "k--", lw=1, label="perfect calibration")
plt.plot(predicted_mean, observed, "o-", color="#55a868", label="specialist")
plt.xlabel("predicted probability"); plt.ylabel("observed frequency")
plt.title("Calibration"); plt.legend()
plt.tight_layout()
plt.savefig("plots/03_calibration.png", dpi=150)
plt.show()


### 3.1 Baseline comparison

A linear model on the same features shows how much the transformer's nonlinearity buys you — the
infinite-slope FS formula involves `tan`, `cos^2`, and multiplicative saturation terms that a linear model
structurally cannot represent.


In [ ]:
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.preprocessing import OneHotEncoder

X_train = pd.concat([
    train_df[NUMERIC_COLS].reset_index(drop=True),
    pd.get_dummies(train_df["soil_texture"], prefix="tex").reset_index(drop=True)
], axis=1)
X_test = pd.concat([
    test_df[NUMERIC_COLS].reset_index(drop=True),
    pd.get_dummies(test_df["soil_texture"], prefix="tex").reset_index(drop=True)
], axis=1).reindex(columns=X_train.columns, fill_value=0)

lin_reg = LinearRegression().fit(X_train, train_df["risk_score"])
lin_pred = lin_reg.predict(X_test)
lin_mae = mean_absolute_error(test_df["risk_score"], lin_pred)
lin_r2 = r2_score(test_df["risk_score"], lin_pred)
lin_auc = roc_auc_score((test_df["risk_score"] >= 0.5).astype(int), lin_pred)

baseline_metrics = {"mae": float(lin_mae), "r2": float(lin_r2), "roc_auc": float(lin_auc)}

fig, axes = plt.subplots(1, 3, figsize=(11, 4))
labels_plot = ["MAE (lower better)", "R\u00b2", "ROC-AUC"]
specialist_vals = [specialist_metrics["mae"], specialist_metrics["r2"], specialist_metrics["roc_auc"]]
baseline_vals = [baseline_metrics["mae"], baseline_metrics["r2"], baseline_metrics["roc_auc"]]
for ax, label, sv, bv in zip(axes, labels_plot, specialist_vals, baseline_vals):
    ax.bar(["Linear baseline", "FT-Transformer"], [bv, sv], color=["#8c8c8c", "#4c72b0"])
    ax.set_title(label)
plt.tight_layout()
plt.savefig("plots/04_baseline_comparison.png", dpi=150)
plt.show()

print("Linear baseline:", json.dumps(baseline_metrics, indent=2))
print("FT-Transformer :", json.dumps(specialist_metrics, indent=2))


## 4. Save the specialist for the leader to call

Bundles weights + preprocessing stats + an inference function with the exact signature the leader LLM
will invoke as a tool.


In [ ]:
torch.save({
    "state_dict": model.state_dict(),
    "num_mean": num_mean.to_dict(),
    "num_std": num_std.to_dict(),
    "texture_to_idx": texture_to_idx,
    "class_names": CLASS_NAMES,
    "numeric_cols": NUMERIC_COLS,
}, "artifacts/landslide_specialist.pt")


def predict_landslide_risk(slope_deg: float, soil_texture: str, soil_depth_m: float,
                            saturation_fraction: float, rain_1h_mmh: float, rain_3h_mmh: float,
                            rain_6h_mmh: float, rain_12h_mmh: float, rain_24h_mmh: float) -> dict:
    """Inference function matching the tool schema the leader LLM will call."""
    row = pd.DataFrame([{
        "slope_deg": slope_deg, "soil_depth_m": soil_depth_m,
        "saturation_fraction": saturation_fraction,
        "rain_1h_mmh": rain_1h_mmh, "rain_3h_mmh": rain_3h_mmh, "rain_6h_mmh": rain_6h_mmh,
        "rain_12h_mmh": rain_12h_mmh, "rain_24h_mmh": rain_24h_mmh,
    }])
    numeric = torch.tensor(((row[NUMERIC_COLS] - num_mean) / num_std).values.astype("float32")).to(DEVICE)
    texture_idx = torch.tensor([texture_to_idx.get(soil_texture, texture_to_idx["Loam"])], dtype=torch.long).to(DEVICE)

    model.eval()
    with torch.no_grad():
        pred_score, pred_logits = model(numeric, texture_idx)
    risk_score = float(pred_score.item())
    risk_class = CLASS_NAMES[int(pred_logits.argmax(-1).item())]
    confidence = float(torch.softmax(pred_logits, dim=-1).max().item())
    return {"risk_score": round(risk_score, 3), "risk_class": risk_class, "confidence": round(confidence, 3)}


# sanity check
print(predict_landslide_risk(
    slope_deg=32, soil_texture="Clay loam", soil_depth_m=1.2, saturation_fraction=0.8,
    rain_1h_mmh=25, rain_3h_mmh=15, rain_6h_mmh=10, rain_12h_mmh=7, rain_24h_mmh=5))


## 5. Leader LLM — tool-calling orchestration demo

The leader calls `predict_landslide_risk` as a tool, then writes a fused explanation. Do **not** hardcode
your API key in this cell — set it as a Colab secret (key icon in the left sidebar) named
`OPENAI_API_KEY`, or export it as an environment variable before running.


In [ ]:
%pip install --quiet openai

import os
from openai import OpenAI

try:
    from google.colab import userdata
    os.environ.setdefault("OPENAI_API_KEY", userdata.get("OPENAI_API_KEY"))
except Exception:
    pass  # not running in Colab, or secret not set - falls back to an already-exported env var

client = OpenAI()  # reads OPENAI_API_KEY from the environment

TOOL_SCHEMA = [{
    "type": "function",
    "function": {
        "name": "predict_landslide_risk",
        "description": "Predict landslide risk from terrain, soil, and rainfall features using the trained specialist model.",
        "parameters": {
            "type": "object",
            "properties": {
                "slope_deg": {"type": "number"},
                "soil_texture": {"type": "string", "enum": TEXTURE_NAMES},
                "soil_depth_m": {"type": "number"},
                "saturation_fraction": {"type": "number"},
                "rain_1h_mmh": {"type": "number"},
                "rain_3h_mmh": {"type": "number"},
                "rain_6h_mmh": {"type": "number"},
                "rain_12h_mmh": {"type": "number"},
                "rain_24h_mmh": {"type": "number"},
            },
            "required": ["slope_deg", "soil_texture", "soil_depth_m", "saturation_fraction",
                          "rain_1h_mmh", "rain_3h_mmh", "rain_6h_mmh", "rain_12h_mmh", "rain_24h_mmh"],
        },
    },
}]

SYSTEM_PROMPT = (
    "You are the leader model in a hazard-assessment system. When asked about landslide risk for a "
    "location, call predict_landslide_risk with the given features, then write a short, concrete "
    "explanation of the result for a first responder. Always state the exact risk_score and risk_class "
    "the tool returned - never round or invent a different number. If useful, note which input factor "
    "(rainfall intensity, slope stability, or soil saturation) is driving the result."
)


def run_leader(user_message: str) -> dict:
    messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_message}]
    response = client.chat.completions.create(
        model="gpt-4o-mini", messages=messages, tools=TOOL_SCHEMA, temperature=0.2)
    msg = response.choices[0].message
    transcript = {"user_message": user_message, "tool_calls": [], "final_answer": None}

    if msg.tool_calls:
        messages.append(msg)
        for call in msg.tool_calls:
            args = json.loads(call.function.arguments)
            result = predict_landslide_risk(**args)
            transcript["tool_calls"].append({"arguments": args, "result": result})
            messages.append({"role": "tool", "tool_call_id": call.id, "content": json.dumps(result)})

        final = client.chat.completions.create(model="gpt-4o-mini", messages=messages, temperature=0.2)
        transcript["final_answer"] = final.choices[0].message.content
    else:
        transcript["final_answer"] = msg.content

    return transcript


SCENARIOS = [
    "Steep hillside above a village: slope 34 degrees, clay loam soil, 1.1m regolith depth, "
    "soil already 85% saturated from days of rain. Last 24h: 90mm total, with a burst of 30mm in the last hour.",
    "Gentle slope, sandy soil, dry conditions: slope 8 degrees, sand, 1.3m depth, 15% saturation, "
    "light scattered rain totalling under 5mm over the last 24 hours.",
]

transcripts = [run_leader(s) for s in SCENARIOS]
for t in transcripts:
    print("=" * 80)
    print("SCENARIO:", t["user_message"])
    print("-" * 80)
    print("TOOL CALL(S):", json.dumps(t["tool_calls"], indent=2))
    print("-" * 80)
    print("LEADER EXPLANATION:", t["final_answer"])

with open("artifacts/leader_transcripts.json", "w") as f:
    json.dump(transcripts, f, indent=2)


## 6. Export everything needed for the results deck

In [ ]:
summary = {
    "n_train": len(train_df), "n_val": len(val_df), "n_test": len(test_df),
    "specialist_param_count": n_params,
    "specialist_metrics": specialist_metrics,
    "linear_baseline_metrics": baseline_metrics,
}
with open("artifacts/results_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))

import shutil
shutil.make_archive("landslide_poc_results", "zip", ".", "plots")
shutil.make_archive("landslide_poc_artifacts", "zip", ".", "artifacts")
print("Download plots/ and artifacts/ (or the two zip files) - send results_summary.json, "
      "the plots/*.png, and artifacts/leader_transcripts.json back to build the results deck.")


## Notes / next steps

- This specialist is one of the *N per-hazard specialists* in the planned architecture — the same
  recipe (physics-grounded synthetic data \u2192 FT-Transformer) applies to flood, wildfire, etc.
- The leader here is a prompted `gpt-4o-mini` for the demo. The production plan is a fine-tuned small
  open model (Qwen2.5-3B / Llama-3.2-3B) via QLoRA on a T4, once zero-shot fusion quality needs improving.
- Real landslide inventories (NASA COOLR, USGS) should replace/augment the synthetic data before this
  goes anywhere near production — this notebook's dataset is a physics-grounded stand-in, not ground truth.
